<a href="https://colab.research.google.com/github/AnjaliAleti/Aleti_INFO5731_Fall2024/blob/main/In_Class_Exercise_5%266_Feature_Extraction_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# In-Class Assignment — Python for Feature Extraction
**Time:** 20 minutes  |  **Points:** 20


....


Dataset file: `product_reviews.txt`

## Load the dataset
Download it from **Canvas**, then run the upload cell below to select the file from your computer.


In [16]:
# Upload the dataset file from your computer (Google Colab)
from google.colab import files

uploaded = files.upload()   # choose product_reviews.txt
filename = next(iter(uploaded))
print("product_reviews.txt:", filename)


Saving product_reviews.txt to product_reviews (4).txt
product_reviews.txt: product_reviews (4).txt


## Q1 (2 point) — Load & Read & inspect

## Note about the header
**Important:** The dataset file includes a **header row** (column names).  
Make sure the header is used as the column names — it **should NOT appear as a data row** in your DataFrame.

Tip: Use the filename variable printed above when reading the file.
After Reading, check that your columns are `id` and `text`.
Print: **(a)** `df.shape`, **(b)** `df.head(3\)`.  


In [17]:
# Write your Answer here (Q1)
import pandas as pd

filename = "product_reviews.txt"

# Read file (header row should be used as column names)
df = pd.read_csv(filename, sep="\t", header=0)

# Check columns
print(df.columns)

# Print shape
print(df.shape)

# Print first 3 rows
print(df.head(3))



Index(['id', 'text'], dtype='object')
(10, 2)
   id                                               text
0   1  Love this blender! Smoothies are super creamy ...
1   2  Terrible quality... stopped working after 2 da...
2   3      Good value for the price. Shipping was quick.


## Q2 (4 points) — Basic handcrafted features  
Create these columns and then display the DataFrame:
- `word_count` = number of words  
- `char_count` = number of characters  
- `avg_word_len` = average word length (ignore punctuation)  
- `excl_count` = number of `!` characters  

Print: `id, word_count, char_count, avg_word_len, excl_count`.


In [18]:
# Write your Answer here (Q2)
import pandas as pd
import re

filename = "product_reviews.txt"

# Read file with header
df = pd.read_csv(filename, sep="\t", header=0)

# --- Basic handcrafted features ---

# number of words
df["word_count"] = df["text"].apply(lambda x: len(x.split()))

# number of characters
df["char_count"] = df["text"].apply(len)

# average word length (ignore punctuation)
def avg_word_length(text):
    words = re.findall(r"\b\w+\b", text)  # keeps only words, removes punctuation
    if len(words) == 0:
        return 0
    return sum(len(w) for w in words) / len(words)

df["avg_word_len"] = df["text"].apply(avg_word_length)

# number of ! characters
df["excl_count"] = df["text"].apply(lambda x: x.count("!"))

# Print required columns
print(df[["id", "word_count", "char_count", "avg_word_len", "excl_count"]])


   id  word_count  char_count  avg_word_len  excl_count
0   1           9          55      5.000000           1
1   2           7          51      5.571429           3
2   3           8          45      4.500000           0
3   4          10          56      4.500000           0
4   5           8          53      5.500000           1
5   6          10          51      4.000000           0
6   7           9          50      4.444444           0
7   8           6          40      5.500000           0
8   9           7          52      6.285714           0
9  10           7          48      4.750000           0


## Q3 (6 points) — Bag-of-Words (CountVectorizer)  
Use `CountVectorizer(stop_words="english")` on `df["text"]`. Print:
1) vocabulary size (number of features)  
2) top 10 words by **total count** across all documents (word + count)


In [19]:
# # Write your Answer here (Q3)
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
import numpy as np

filename = "product_reviews.txt"

# Read dataset
df = pd.read_csv(filename, sep="\t", header=0)

# --- Bag of Words ---
vectorizer = CountVectorizer(stop_words="english")

X = vectorizer.fit_transform(df["text"])

# vocabulary size
vocab_size = len(vectorizer.get_feature_names_out())
print("Vocabulary size:", vocab_size)

# total count for each word
word_counts = np.asarray(X.sum(axis=0)).flatten()

words = vectorizer.get_feature_names_out()

# combine words + counts
word_freq = list(zip(words, word_counts))

# sort by count (descending)
word_freq_sorted = sorted(word_freq, key=lambda x: x[1], reverse=True)

# top 10 words
print("\nTop 10 words:")
for w, c in word_freq_sorted[:10]:
    print(w, c)


Vocabulary size: 50

Top 10 words:
amazing 1
app 1
battery 1
blender 1
box 1
buy 1
charged 1
clear 1
crashing 1
creamy 1


## Q4 (4 points) — Bigram features  
Use `CountVectorizer(stop_words="english", ngram_range=(2,2))`.  
Print the top 5 bigrams by total count (bigram + count).


In [20]:
# Write your Answer here (Q4)
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
import numpy as np

filename = "product_reviews.txt"

# Read dataset
df = pd.read_csv(filename, sep="\t", header=0)

# --- Bigram features ---
vectorizer = CountVectorizer(stop_words="english", ngram_range=(2, 2))

X = vectorizer.fit_transform(df["text"])

bigrams = vectorizer.get_feature_names_out()

# total counts for each bigram
counts = np.asarray(X.sum(axis=0)).flatten()

bigram_counts = list(zip(bigrams, counts))

# sort by count descending
bigram_counts = sorted(bigram_counts, key=lambda x: x[1], reverse=True)

# print top 5
print("Top 5 bigrams:")
for bg, c in bigram_counts[:5]:
    print(bg, c)


Top 5 bigrams:
amazing screen 1
app keeps 1
battery life 1
blender smoothies 1
box damaged 1


## Q5 (4 points) — TF-IDF features  
Use `TfidfVectorizer(stop_words="english", ngram_range=(1,2))`.  
Compute the **average TF-IDF** score of each term across documents and print the top 5 terms (term + avg score).


In [21]:
# Write your Answer here  (Q5)
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

filename = "product_reviews.txt"

# Read dataset
df = pd.read_csv(filename, sep="\t", header=0)

# --- TF-IDF features ---
vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2))

X = vectorizer.fit_transform(df["text"])

terms = vectorizer.get_feature_names_out()

# average TF-IDF score for each term
avg_tfidf = np.asarray(X.mean(axis=0)).flatten()

term_scores = list(zip(terms, avg_tfidf))

# sort descending
term_scores = sorted(term_scores, key=lambda x: x[1], reverse=True)

# top 5 terms
print("Top 5 TF-IDF terms:")
for t, s in term_scores[:5]:
    print(t, round(s, 4))



Top 5 TF-IDF terms:
clear 0.0378
easy 0.0378
easy instructions 0.0378
instructions 0.0378
instructions clear 0.0378


## Grading Checklist
- Q1: correct prints  
- Q2: correct feature columns + requested display  
- Q3: correct vocabulary size + correct top 10 words by total count  
- Q4: correct top 5 bigrams by total count  
- Q5: correct top 5 TF-IDF terms by average score
